Bài tập 1 - Bayes với ba nguyên nhân

In [1]:
def bayes_ba_nhanh(tien_nghiem, kha_nang):
    """Tinh xac suat hau nghiem cho ba nhom nguyen nhan."""
    mau_so = sum(t * k for t, k in zip(tien_nghiem, kha_nang))
    return [(t * k) / mau_so for t, k in zip(tien_nghiem, kha_nang)]

nhom = ["Hoc deu", "Hoc don", "It hoc"]
tien_nghiem = [0.50, 0.30, 0.20]
kha_nang = [0.95, 0.70, 0.30]

hau_nghiem = bayes_ba_nhanh(tien_nghiem, kha_nang)

for ten, p in zip(nhom, hau_nghiem):
    print(f"Xac suat ban do thuoc nhom {ten:7s}: {p:.4f}")

print()
print(f"Tong ba xac suat: {sum(hau_nghiem):.4f}")
print("Nhan xet: Tong ba xac suat bang dung 1.0000. Ban qua mon co kha nang cao nhat thuoc nhom Hoc deu (63.76%) vi nhom nay vua chiem dong dao vua co ty le qua mon vuot troi.")

Xac suat ban do thuoc nhom Hoc deu: 0.6376
Xac suat ban do thuoc nhom Hoc don: 0.2819
Xac suat ban do thuoc nhom It hoc : 0.0805

Tong ba xac suat: 1.0000
Nhan xet: Tong ba xac suat bang dung 1.0000. Ban qua mon co kha nang cao nhat thuoc nhom Hoc deu (63.76%) vi nhom nay vua chiem dong dao vua co ty le qua mon vuot troi.


Bài tập 2 - Từ nào tố cáo mạnh nhất

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

mo_hinh = MultinomialNB()
mo_hinh.fit(X, y)

tu = vec.get_feature_names_out()
hieu = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]

df_tu = pd.DataFrame({"tu": tu, "hieu": hieu})

print("5 tu to cao tin rac manh nhat (hieu loga duong lon nhat):")
for _, r in df_tu.sort_values("hieu", ascending=False).head(5).iterrows():
    print(f"  {r.tu:10s}: {r.hieu:+.4f}")

print()
print("5 tu cho thay tin thuong manh nhat (hieu loga am lon nhat):")
for _, r in df_tu.sort_values("hieu", ascending=True).head(5).iterrows():
    print(f"  {r.tu:10s}: {r.hieu:+.4f}")

5 tu to cao tin rac manh nhat (hieu loga duong lon nhat):
  mãi       : +3.2034
  tay       : +2.9056
  ngay      : +2.6618
  miễn      : +2.4795
  trúng     : +2.3742

5 tu cho thay tin thuong manh nhat (hieu loga am lon nhat):
  lớp       : -2.6122
  em        : -2.5618
  bài       : -2.2946
  báo       : -2.2741
  phòng     : -2.2741


Bài tập 3 - Đổi tham số làm trơn

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print(f"{'alpha':>7s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s}")
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    mo_hinh = MultinomialNB(alpha=alpha)
    mo_hinh.fit(X_train, y_train)
    y_pred = mo_hinh.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    print(f"{alpha:7.3f} {acc:10.4f} {pre:10.4f} {rec:10.4f}")

print()
print("Nhan xet: Voi tap du lieu nay, cac tu to cao rat ro ret nen trong khoang alpha tu 0.001 toi 100.0 ket qua du doan van giu nguyen. Neu alpha tang len cuc lon (vi du 1000), tham so lam tron se at han so dem thuc te, keo xac suat moi tu xich lai gan nhau ve muc 1/V va triet tieu bang chung; mo hinh khi do chi con dua vao tien nghiem de doan tat ca la lop thuong (lop da so), lam Recall cua lop rac roi thang xuong 0.")

  alpha   Accuracy  Precision     Recall
  0.001     0.9467     1.0000     0.8667
  0.100     0.9467     1.0000     0.8667
  1.000     0.9467     1.0000     0.8667
 10.000     0.9467     1.0000     0.8667
100.000     0.9467     1.0000     0.8667

Nhan xet: Voi tap du lieu nay, cac tu to cao rat ro ret nen trong khoang alpha tu 0.001 toi 100.0 ket qua du doan van giu nguyen. Neu alpha tang len cuc lon (vi du 1000), tham so lam tron se at han so dem thuc te, keo xac suat moi tu xich lai gan nhau ve muc 1/V va triet tieu bang chung; mo hinh khi do chi con dua vao tien nghiem de doan tat ca la lop thuong (lop da so), lam Recall cua lop rac roi thang xuong 0.


Bài tập 4 - Tự viết bộ lọc không dùng thư viện

In [4]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17, stratify=df["la_rac"]
)

# 1. Mo hinh thu vien
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train_vec = vec.fit_transform(X_train_raw)
X_test_vec = vec.transform(X_test_raw)

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train_vec, y_train)
acc_sklearn = accuracy_score(y_test, mo_hinh.predict(X_test_vec))

# 2. Bo loc tu viet bang tay
tu_dien = sorted({t for cau in X_train_raw for t in cau.split()})
V = len(tu_dien)

dem = {0: Counter(), 1: Counter()}
tong_tu = {0: 0, 1: 0}
so_tin = {0: (y_train == 0).sum(), 1: (y_train == 1).sum()}

for cau, nhan in zip(X_train_raw, y_train):
    for t in cau.split():
        dem[nhan][t] += 1
        tong_tu[nhan] += 1

alpha = 1.0
log_prior_0 = np.log(so_tin[0] / len(y_train))
log_prior_1 = np.log(so_tin[1] / len(y_train))

y_pred_tu_viet = []
for cau in X_test_raw:
    diem_0 = log_prior_0
    diem_1 = log_prior_1
    for t in cau.split():
        if t in tu_dien:
            diem_0 += np.log((dem[0][t] + alpha) / (tong_tu[0] + alpha * V))
            diem_1 += np.log((dem[1][t] + alpha) / (tong_tu[1] + alpha * V))
    y_pred_tu_viet.append(1 if diem_1 > diem_0 else 0)

acc_tu_viet = accuracy_score(y_test, y_pred_tu_viet)

print(f"Accuracy scikit-learn : {acc_sklearn:.4f}")
print(f"Accuracy bo loc tu viet: {acc_tu_viet:.4f}")
print()
print("Giai thich: Hai con so accuracy khop hoan toan (cung dat 0.9467) vi bo loc tu viet da tai hien chuan xac co che cua MultinomialNB: lay log xac suat tien nghiem tu ty le lop, ap dung lam tron Laplace alpha = 1.0 cho xac suat moi tu roi cong don loga de tranh hien tuong tran duoi so hoc.")

Accuracy scikit-learn : 0.9467
Accuracy bo loc tu viet: 0.9467

Giai thich: Hai con so accuracy khop hoan toan (cung dat 0.9467) vi bo loc tu viet da tai hien chuan xac co che cua MultinomialNB: lay log xac suat tien nghiem tu ty le lop, ap dung lam tron Laplace alpha = 1.0 cho xac suat moi tu roi cong don loga de tranh hien tuong tran duoi so hoc.


Bài tập 5 - Chọn sai biến thể Naive Bayes

In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh_gnb = GaussianNB().fit(X_train, y_train)
acc_gnb = accuracy_score(y_test, mo_hinh_gnb.predict(X_test))

mo_hinh_mnb = MultinomialNB().fit(X_train, y_train)
acc_mnb = accuracy_score(y_test, mo_hinh_mnb.predict(X_test))

print(f"Accuracy GaussianNB   : {acc_gnb:.4f}")
print(f"Accuracy MultinomialNB: {acc_mnb:.4f}")
print()
print("Tra loi cau hoi:")
print("- Chuong trinh KHONG bao loi vi cac gia tri trong gio_on va diem_giua_ky deu la so duong (MultinomialNB chi dung chay neu du lieu co so am).")
print(f"- Con so accuracy dat {acc_mnb:.4f}, thap hon muc {acc_gnb:.4f} cua GaussianNB.")
print("- Giai thich: Dung MultinomialNB o day la sai ve nguyen tac vi no duoc thiet ke cho bien dem roi rac (nhu so lan xuat hien tu). O day gio_on va diem_giua_ky la cac dai luong so thuc lien tuc; viec mo hinh hoa chung bang duong chuong phan phoi chuan qua GaussianNB moi phan anh dung ban chat vat ly cua du lieu.")

Accuracy GaussianNB   : 0.9000
Accuracy MultinomialNB: 0.8000

Tra loi cau hoi:
- Chuong trinh KHONG bao loi vi cac gia tri trong gio_on va diem_giua_ky deu la so duong (MultinomialNB chi dung chay neu du lieu co so am).
- Con so accuracy dat 0.8000, thap hon muc 0.9000 cua GaussianNB.
- Giai thich: Dung MultinomialNB o day la sai ve nguyen tac vi no duoc thiet ke cho bien dem roi rac (nhu so lan xuat hien tu). O day gio_on va diem_giua_ky la cac dai luong so thuc lien tuc; viec mo hinh hoa chung bang duong chuong phan phoi chuan qua GaussianNB moi phan anh dung ban chat vat ly cua du lieu.


Bài tập 6 - Lớp hiếm và xác suất tiên nghiệm

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

df_thuong = df[df["la_rac"] == 0]
df_rac = df[df["la_rac"] == 1].head(20)
df_lech = pd.concat([df_thuong, df_rac]).reset_index(drop=True)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df_lech["noi_dung"])
y = df_lech["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = MultinomialNB().fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

acc = accuracy_score(y_test, y_pred)
rec_rac = recall_score(y_test, y_pred, pos_label=1)

print(f"Accuracy       : {acc:.4f}")
print(f"Recall lop rac : {rec_rac:.4f}")
print()
print("Nhan xet: Accuracy khong con phan anh dung chat luong bo loc vi ta da vuong vao bay accuracy tren du lieu mat can bang. Do tin thuong chiem toi 90% tap du lieu (45 tren 50 tin kiem tra), mo hinh chi can doan dung lop thuong la accuracy da de dang vuot 90% (o day dat 98.00%). Tuy nhien, khi nhin vao muc tieu chinh la chan tin rac, mo hinh da bo sot 1 tren tong so 5 tin rac kiem tra khien Recall tut xuong 80.00%.")

Accuracy       : 0.9800
Recall lop rac : 0.8000

Nhan xet: Accuracy khong con phan anh dung chat luong bo loc vi ta da vuong vao bay accuracy tren du lieu mat can bang. Do tin thuong chiem toi 90% tap du lieu (45 tren 50 tin kiem tra), mo hinh chi can doan dung lop thuong la accuracy da de dang vuot 90% (o day dat 98.00%). Tuy nhien, khi nhin vao muc tieu chinh la chan tin rac, mo hinh da bo sot 1 tren tong so 5 tin rac kiem tra khien Recall tut xuong 80.00%.
